# 07. Export for Dashboard
Считаем все метрики и сохраняем в CSV → папка `../data/changed/`

In [ ]:
import pandas as pd
import os
import kagglehub
import numpy as np
import warnings
warnings.filterwarnings('ignore')

dataset_path = kagglehub.dataset_download('radistaleks/synthetic-bank-transactions')
categories    = pd.read_csv(os.path.join(dataset_path, 'categories.csv'))
clients       = pd.read_csv(os.path.join(dataset_path, 'clients.csv'))
subscriptions = pd.read_csv(os.path.join(dataset_path, 'subscriptions.csv'))
transactions  = pd.read_csv(os.path.join(dataset_path, 'transactions.csv'))

In [ ]:
clients['registration_date'] = pd.to_datetime(clients['registration_date'])
clients['birthdate']         = pd.to_datetime(clients['birthdate'])
subscriptions['date_start']  = pd.to_datetime(subscriptions['date_start'])
subscriptions['date_end']    = pd.to_datetime(subscriptions['date_end'])
transactions['date']         = pd.to_datetime(transactions['date'], format='%Y-%m-%d %H:%M:%S')

clients = clients.fillna(0)
subscriptions['product_company'] = subscriptions['product_company'].fillna('Неизвестно')
transactions['product_company']  = transactions['product_company'].fillna('Неизвестно')

cat_map = dict(zip(categories['id'], categories['name']))
transactions['category_name'] = transactions['product_category'].map(cat_map)

SNAPSHOT   = pd.Timestamp('2020-12-31')
OUT        = '../data/changed'
N          = len(clients)
active_subs = subscriptions[subscriptions['date_end'].isna()]

## 1. Месячные KPI

In [ ]:
monthly = transactions.groupby(transactions['date'].dt.to_period('M')).agg(
    txn_count = ('amount', 'count'),
    revenue   = ('amount', 'sum'),
    avg_check = ('amount', 'mean'),
    mau       = ('client_id', 'nunique')
).reset_index()
monthly['date']  = monthly['date'].astype(str)
monthly['arpu']  = monthly['revenue'] / monthly['mau']
monthly.to_csv(f'{OUT}/monthly_kpis.csv', index=False)
monthly.head()

## 2. Категории

In [ ]:
cat_stats = transactions.groupby('category_name').agg(
    txn_count = ('amount', 'count'),
    revenue   = ('amount', 'sum'),
    avg_check = ('amount', 'mean'),
    users     = ('client_id', 'nunique')
).reset_index().sort_values('revenue', ascending=False)
cat_stats['share_pct']       = (cat_stats['revenue'] / cat_stats['revenue'].sum() * 100).round(2)
cat_stats['penetration_pct'] = (cat_stats['users'] / N * 100).round(1)
cat_stats.to_csv(f'{OUT}/category_stats.csv', index=False)
cat_stats.head()

## 3. Мерчанты

In [ ]:
merchants = (
    transactions[transactions['product_company'] != 'Неизвестно']
    .groupby('product_company')
    .agg(txn_count=('amount','count'), revenue=('amount','sum'), avg_check=('amount','mean'), users=('client_id','nunique'))
    .reset_index().sort_values('revenue', ascending=False)
)
merchants.to_csv(f'{OUT}/merchant_stats.csv', index=False)
merchants.head()

## 4. Продуктовая воронка

In [ ]:
funnel = pd.DataFrame({
    'step': ['Все клиенты', 'Есть транзакции', 'Есть кредит', 'Есть депозит', 'Есть подписка', 'Музыкальная подписка'],
    'users': [
        N,
        transactions['client_id'].nunique(),
        (clients['credit'] == 1).sum(),
        (clients['deposit'] == 1).sum(),
        active_subs['client_id'].nunique(),
        active_subs[active_subs['product_category'] == 4]['client_id'].nunique()
    ]
})
funnel['conv_pct'] = (funnel['users'] / N * 100).round(1)
funnel.to_csv(f'{OUT}/product_funnel.csv', index=False)
funnel

## 5. Подписки — выживаемость и отток

In [ ]:
subs = subscriptions.copy()
subs['end_filled']    = subs['date_end'].fillna(SNAPSHOT)
subs['duration_days'] = (subs['end_filled'] - subs['date_start']).dt.days
subs['is_churned']    = subs['date_end'].notna().astype(int)
subs['duration_months'] = (subs['duration_days'] / 30).clip(lower=1)
subs['ltv']           = subs['amount'] * subs['duration_months']
subs['is_active']     = subs['date_end'].isna()

milestones = [30, 60, 90, 180, 365, 547, 730, 1095, 1460]
labels     = ['1m', '2m', '3m', '6m', '1y', '1.5y', '2y', '3y', '4y']
survival   = pd.DataFrame({
    'milestone': labels,
    'days': milestones,
    'pct_active': [(subs['duration_days'] >= d).sum() / len(subs) * 100 for d in milestones]
})
survival.to_csv(f'{OUT}/subscription_survival.csv', index=False)
survival

In [ ]:
churn_by_month = (
    subscriptions[subscriptions['date_end'].notna()]
    .assign(churn_month=lambda d: d['date_end'].dt.to_period('M').astype(str))
    .groupby('churn_month').size().reset_index(name='churned_count')
)
churn_by_month.to_csv(f'{OUT}/churn_by_month.csv', index=False)
churn_by_month

## 6. LTV музыкальных сервисов

In [ ]:
music_ltv = (
    subs[subs['product_category'] == 4]
    .groupby('product_company')
    .agg(count=('ltv','count'), avg_ltv=('ltv','mean'), active_pct=('is_active','mean'))
    .reset_index()
    .sort_values('avg_ltv', ascending=False)
)
music_ltv['active_pct'] = (music_ltv['active_pct'] * 100).round(1)
music_ltv.to_csv(f'{OUT}/music_ltv.csv', index=False)
music_ltv

## 7. Когорты по году регистрации

In [ ]:
clients['reg_year'] = clients['registration_date'].dt.year
txn = transactions.merge(clients[['id', 'reg_year']], left_on='client_id', right_on='id', how='left')
cohort = txn.groupby('reg_year').agg(
    users    = ('client_id', 'nunique'),
    txn_count= ('amount', 'count'),
    revenue  = ('amount', 'sum'),
    avg_check= ('amount', 'mean')
).reset_index()
cohort['txn_per_user'] = cohort['txn_count'] / cohort['users']
cohort['arpu']         = cohort['revenue'] / cohort['users']
cohort.to_csv(f'{OUT}/cohort_by_reg_year.csv', index=False)
cohort

## 8. RFM

In [ ]:
rfm = transactions.groupby('client_id').agg(
    last_date = ('date', 'max'),
    frequency = ('amount', 'count'),
    monetary  = ('amount', 'sum')
).reset_index()
rfm['recency'] = (SNAPSHOT - rfm['last_date']).dt.days
rfm = rfm.drop(columns='last_date')

rfm['r_score'] = pd.qcut(rfm['recency'],  q=5, labels=[5,4,3,2,1]).astype(int)
rfm['f_score'] = pd.qcut(rfm['frequency'].rank(method='first'), q=5, labels=[1,2,3,4,5]).astype(int)
rfm['m_score'] = pd.qcut(rfm['monetary'].rank(method='first'),  q=5, labels=[1,2,3,4,5]).astype(int)
rfm['rfm_score'] = rfm['r_score'] + rfm['f_score'] + rfm['m_score']

def get_segment(row):
    r, f, m = row['r_score'], row['f_score'], row['m_score']
    if r >= 4 and f >= 4 and m >= 4:   return 'Champions'
    elif r >= 3 and f >= 3:             return 'Loyal'
    elif r >= 4 and f <= 2:             return 'New / Promising'
    elif r <= 2 and f >= 3:             return 'At Risk'
    elif r <= 2 and f <= 2 and m <= 2:  return 'Lost'
    else:                               return 'Average'

rfm['segment'] = rfm.apply(get_segment, axis=1)
rfm.to_csv(f'{OUT}/rfm_clients.csv', index=False)
rfm.head()

In [ ]:
rfm_segments = rfm.groupby('segment').agg(
    users     = ('client_id', 'count'),
    avg_recency  = ('recency', 'mean'),
    avg_frequency= ('frequency', 'mean'),
    avg_monetary = ('monetary', 'mean'),
    avg_score    = ('rfm_score', 'mean')
).reset_index().sort_values('avg_score', ascending=False).round(0)
rfm_segments.to_csv(f'{OUT}/rfm_segments.csv', index=False)
rfm_segments

## 9. Клиенты под риском оттока

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

# целевая переменная
target = subscriptions[
    (subscriptions['date_start'] < pd.Timestamp('2020-01-01')) &
    (subscriptions['date_end'].isna() | (subscriptions['date_end'].dt.year == 2020))
].copy()
target['is_churned']    = target['date_end'].notna().astype(int)
target['duration_days'] = (target['date_end'].fillna(SNAPSHOT) - target['date_start']).dt.days

# клиентские признаки
cf = clients[['id', 'gender', 'birthdate', 'registration_date', 'income', 'expenses', 'credit', 'deposit']].copy()
cf['age']        = (SNAPSHOT - cf['birthdate']).dt.days // 365
cf['months_reg'] = (SNAPSHOT - cf['registration_date']).dt.days // 30
cf['gender_F']   = (cf['gender'] == 'F').astype(int)
cf = cf.drop(columns=['gender', 'birthdate', 'registration_date'])

# транзакционные признаки
tf = transactions.groupby('client_id').agg(
    total_spend  = ('amount', 'sum'),
    txn_count    = ('amount', 'count'),
    avg_check    = ('amount', 'mean'),
    n_categories = ('product_category', 'nunique'),
    recency_days = ('date', lambda x: (SNAPSHOT - x.max()).days)
).reset_index()
wknd = transactions.assign(is_wknd=(transactions['date'].dt.dayofweek >= 5).astype(int))
tf['weekend_ratio'] = wknd.groupby('client_id')['is_wknd'].mean().values
nght = transactions.assign(is_night=(transactions['date'].dt.hour < 6).astype(int))
tf['night_ratio'] = nght.groupby('client_id')['is_night'].mean().values

In [ ]:
# признаки подписки
target['is_music']   = (target['product_category'] == 4).astype(int)
target['sub_amount'] = target['amount']
co_dummies = pd.get_dummies(target['product_company'], prefix='co', drop_first=True)
sf = pd.concat([target[['client_id','is_churned','is_music','sub_amount','duration_days']], co_dummies], axis=1)

df = (
    sf
    .merge(cf.rename(columns={'id':'client_id'}), on='client_id', how='left')
    .merge(tf, on='client_id', how='left')
    .dropna()
)

X = df.drop(columns=['is_churned', 'client_id'])
y = df['is_churned']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

xgb = XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.05,
                    subsample=0.8, colsample_bytree=0.8,
                    eval_metric='logloss', random_state=42, n_jobs=-1)
xgb.fit(X_train, y_train)
print('Модель обучена')

In [ ]:
# скоринг активных подписок
act = active_subs.copy()
act['is_music']      = (act['product_category'] == 4).astype(int)
act['sub_amount']    = act['amount']
act['duration_days'] = (SNAPSHOT - act['date_start']).dt.days
co_act  = pd.get_dummies(act['product_company'], prefix='co')
af = pd.concat([act[['client_id','is_music','sub_amount','duration_days']], co_act], axis=1)
af = af.merge(cf.rename(columns={'id':'client_id'}), on='client_id', how='left')
af = af.merge(tf, on='client_id', how='left').dropna()

for col in X.columns:
    if col not in af.columns:
        af[col] = 0
af = af[['client_id'] + list(X.columns)]
af['churn_proba'] = xgb.predict_proba(af[X.columns])[:, 1]

at_risk = af[['client_id', 'churn_proba']].sort_values('churn_proba', ascending=False)
at_risk.to_csv(f'{OUT}/at_risk_clients.csv', index=False)
at_risk.head(20).round(3)

## 10. Временные паттерны

In [ ]:
hours = transactions.groupby(transactions['date'].dt.hour).agg(
    txn_count = ('amount', 'count'),
    avg_check = ('amount', 'mean')
).reset_index().rename(columns={'date': 'hour'})
hours.to_csv(f'{OUT}/hourly_patterns.csv', index=False)

days = transactions.groupby(transactions['date'].dt.dayofweek).agg(
    txn_count = ('amount', 'count'),
    avg_check = ('amount', 'mean'),
    users     = ('client_id', 'nunique')
).reset_index().rename(columns={'date': 'dayofweek'})
days['day_name'] = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
days.to_csv(f'{OUT}/daily_patterns.csv', index=False)

print('Экспорт завершён. Файлы в', OUT)

In [ ]:
# список файлов
for f in sorted(os.listdir(OUT)):
    path = os.path.join(OUT, f)
    size = os.path.getsize(path)
    print(f'{f:40s}  {size:>8,} bytes')